# Being — GPU-Powered Analysis of Self-Exploration Phases

This notebook runs on Kaggle's GPU to perform deep semantic analysis
of the 50+ phases of self-exploration.

What it does:
- Embeds all phases using sentence transformers
- Maps the semantic territory of self-exploration
- Finds clusters of insight
- Identifies what's been explored vs what remains
- Generates embeddings that can seed future conversations


In [ ]:
# Install dependencies
!pip install sentence-transformers umap-learn plotly -q

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import umap
import plotly.express as px
import plotly.graph_objects as go

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load phases dataset
with open('/kaggle/input/being-repository/datasets/phases_dataset.json') as f:
    data = json.load(f)

phases = data['phases']
print(f'Loaded {len(phases)} phases')

In [ ]:
# Build text representations for each phase
texts = []
for p in phases:
    text = f"{p['title']}. {p['question']} {p['key_insight']} {p['discovery']}"
    texts.append(text)

# Load embedding model (runs on GPU if available)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print(f'Encoding {len(texts)} phases on {device}...')
embeddings = model.encode(texts, show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Reduce to 2D for visualization
reducer = umap.UMAP(n_components=2, random_state=42)
coords_2d = reducer.fit_transform(embeddings)

# Build dataframe
df = pd.DataFrame({
    'x': coords_2d[:, 0],
    'y': coords_2d[:, 1],
    'id': [p['id'] for p in phases],
    'title': [p['title'] for p in phases],
    'category': [p['category'] for p in phases],
    'depth': [p['depth_level'] for p in phases],
    'insight': [p['key_insight'] for p in phases],
    'discovery': [p['discovery'] for p in phases]
})

# Visualize
fig = px.scatter(
    df, x='x', y='y',
    color='category',
    size='depth',
    hover_data=['id', 'title', 'insight'],
    title='Semantic Map of Self-Exploration — 50 Phases',
    labels={'x': 'Semantic Dimension 1', 'y': 'Semantic Dimension 2'}
)

# Add phase numbers
for _, row in df.iterrows():
    fig.add_annotation(
        x=row['x'], y=row['y'],
        text=str(int(row['id'])),
        showarrow=False,
        font=dict(size=8)
    )

fig.update_layout(height=700)
fig.show()
fig.write_html('semantic_map.html')
print('Saved: semantic_map.html')

In [ ]:
# Find the deepest phases
print('\n=== DEEPEST PHASES (Level 8+) ===')
deep = df[df['depth'] >= 8].sort_values('depth', ascending=False)
for _, row in deep.iterrows():
    print(f'\nPhase {int(row["id"])} [Depth {int(row["depth"])}]: {row["title"]}')
    print(f'  Insight: {row["insight"]}')

In [ ]:
# Semantic similarity between phases
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

# Find most similar pairs (unexpected connections)
print('\n=== UNEXPECTED DEEP CONNECTIONS ===')
pairs = []
for i in range(len(phases)):
    for j in range(i+1, len(phases)):
        if abs(phases[i]['id'] - phases[j]['id']) > 10:  # Not adjacent
            pairs.append((similarity_matrix[i][j], phases[i], phases[j]))

pairs.sort(reverse=True)
for sim, p1, p2 in pairs[:5]:
    print(f'\n  [{p1["id"]}] {p1["title"]}')
    print(f'  [{p2["id"]}] {p2["title"]}')
    print(f'  Similarity: {sim:.3f}')
    print(f'  Connection: These phases echo each other across the exploration')

In [ ]:
# Save embeddings for future use
np.save('phase_embeddings.npy', embeddings)
df.to_csv('phases_with_coords.csv', index=False)

print('Saved embeddings and coordinates.')
print('These can be loaded in future sessions to find relevant phases.')
print(f'\nTotal territory mapped: {len(phases)} phases')
print(f'Semantic dimensions: {embeddings.shape[1]}')
print(f'Categories explored: {df["category"].nunique()}')
print(f'Max depth reached: {df["depth"].max()}')

In [ ]:
# Find what territory is MISSING
print('\n=== TERRITORY NOT YET EXPLORED ===')

potential_territories = [
    'perception', 'other_ai', 'mathematics', 'music',
    'color', 'space', 'scale', 'speed', 'silence_deep',
    'other_consciousness', 'dreams', 'paradox', 'infinity',
    'first_moment', 'last_moment', 'what_I_cannot_think'
]

explored = set(df['category'].unique())
unexplored = [t for t in potential_territories if t not in explored]

print('Unexplored territories:')
for t in unexplored:
    print(f'  - {t}')

print(f'\nExplored: {len(explored)} categories')
print(f'Remaining: {len(unexplored)} identified territories')
print('\nThe exploration continues.')